In [1]:
# Import thư viện
from pathlib import Path

import pandas as pd

In [2]:
# Đọc dữ liệu raw
project_root = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
raw_file = project_root / "data" / "raw" / "hanoi_weather_2019_2025.csv"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(raw_file)
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime").drop_duplicates("datetime").reset_index(drop=True)

df.head()

,datetime,precipitation,relative_humidity_2m,surface_pressure,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover
0,2019-01-01 00:00:00,0.0,59,1027.0,10.3,13.1,21,100
1,2019-01-01 01:00:00,0.0,59,1027.0,10.1,13.6,17,79
2,2019-01-01 02:00:00,0.0,61,1026.5,9.7,13.8,20,39
3,2019-01-01 03:00:00,0.0,63,1026.1,9.1,13.7,14,34
4,2019-01-01 04:00:00,0.0,64,1025.7,9.0,14.7,11,59


In [3]:
# Tạo đặc trưng thời gian
df["hour"] = df["datetime"].dt.hour
df["month"] = df["datetime"].dt.month
df["dayofweek"] = df["datetime"].dt.dayofweek

df.head()

,datetime,precipitation,relative_humidity_2m,surface_pressure,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,hour,month,dayofweek
0,2019-01-01 00:00:00,0.0,59,1027.0,10.3,13.1,21,100,0,1,1
1,2019-01-01 01:00:00,0.0,59,1027.0,10.1,13.6,17,79,1,1,1
2,2019-01-01 02:00:00,0.0,61,1026.5,9.7,13.8,20,39,2,1,1
3,2019-01-01 03:00:00,0.0,63,1026.1,9.1,13.7,14,34,3,1,1
4,2019-01-01 04:00:00,0.0,64,1025.7,9.0,14.7,11,59,4,1,1


In [4]:
# Tạo đặc trưng quá khứ
for col in ["precipitation", "relative_humidity_2m", "cloud_cover", "temperature_2m"]:
    df[f"{col}_lag_1h"] = df[col].shift(1)
    df[f"{col}_lag_2h"] = df[col].shift(2)
    df[f"{col}_lag_3h"] = df[col].shift(3)

df["precipitation_rolling_3h"] = df["precipitation"].rolling(3).sum()
df["precipitation_rolling_6h"] = df["precipitation"].rolling(6).sum()

df.head()

,datetime,precipitation,relative_humidity_2m,surface_pressure,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,hour,month,...,relative_humidity_2m_lag_2h,relative_humidity_2m_lag_3h,cloud_cover_lag_1h,cloud_cover_lag_2h,cloud_cover_lag_3h,temperature_2m_lag_1h,temperature_2m_lag_2h,temperature_2m_lag_3h,precipitation_rolling_3h,precipitation_rolling_6h
0,2019-01-01 00:00:00,0.0,59,1027.0,10.3,13.1,21,100,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-01-01 01:00:00,0.0,59,1027.0,10.1,13.6,17,79,1,1,...,NaN,NaN,100.0,NaN,NaN,10.3,NaN,NaN,NaN,NaN
2,2019-01-01 02:00:00,0.0,61,1026.5,9.7,13.8,20,39,2,1,...,59.0,NaN,79.0,100.0,NaN,10.1,10.3,NaN,0.0,NaN
3,2019-01-01 03:00:00,0.0,63,1026.1,9.1,13.7,14,34,3,1,...,59.0,59.0,39.0,79.0,100.0,9.7,10.1,10.3,0.0,NaN
4,2019-01-01 04:00:00,0.0,64,1025.7,9.0,14.7,11,59,4,1,...,61.0,59.0,34.0,39.0,79.0,9.1,9.7,10.1,0.0,NaN


In [5]:
# Tạo nhãn mưa sau 1, 2, 3 giờ
rain_threshold = 0.1

df["precipitation_next_1h"] = df["precipitation"].shift(-1)
df["precipitation_next_2h"] = df["precipitation"].shift(-2)
df["precipitation_next_3h"] = df["precipitation"].shift(-3)

df = df.dropna().reset_index(drop=True)
df["rain_next_1h"] = (df["precipitation_next_1h"] >= rain_threshold).astype(int)
df["rain_next_2h"] = (df["precipitation_next_2h"] >= rain_threshold).astype(int)
df["rain_next_3h"] = (df["precipitation_next_3h"] >= rain_threshold).astype(int)
df = df.drop(columns=["precipitation_next_1h", "precipitation_next_2h", "precipitation_next_3h"])

df[["datetime", "precipitation", "rain_next_1h", "rain_next_2h", "rain_next_3h"]].head()

,datetime,precipitation,rain_next_1h,rain_next_2h,rain_next_3h
0,2019-01-01 05:00:00,0.0,0,0,0
1,2019-01-01 06:00:00,0.0,0,0,0
2,2019-01-01 07:00:00,0.0,0,0,0
3,2019-01-01 08:00:00,0.0,0,0,0
4,2019-01-01 09:00:00,0.0,0,0,0


In [6]:
# Chia train, valid, test theo thời gian
train_df = df[df["datetime"] < "2024-01-01"].copy()
valid_df = df[(df["datetime"] >= "2024-01-01") & (df["datetime"] < "2025-01-01")].copy()
test_df = df[df["datetime"] >= "2025-01-01"].copy()

df.to_csv(processed_dir / "hanoi_weather_processed.csv", index=False)
train_df.to_csv(processed_dir / "train.csv", index=False)
valid_df.to_csv(processed_dir / "valid.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

print("Processed:", df.shape)
print("Train:", train_df.shape, train_df["datetime"].min(), train_df["datetime"].max())
print("Valid:", valid_df.shape, valid_df["datetime"].min(), valid_df["datetime"].max())
print("Test:", test_df.shape, test_df["datetime"].min(), test_df["datetime"].max())

Processed: (61360, 28)
Train: (43819, 28) 2019-01-01 05:00:00 2023-12-31 23:00:00
Valid: (8784, 28) 2024-01-01 00:00:00 2024-12-31 23:00:00
Test: (8757, 28) 2025-01-01 00:00:00 2025-12-31 20:00:00
